# Notebook 2. Round-to-nearest (RTN)

Round-to-nearest applies the quantize and dequantize steps of Notebook 1 to every linear layer of a model, with no calibration data and no optimization. It is the baseline that GPTQ, AWQ and SmoothQuant improve on, and its storage format, packed INT4 codes with per-group scales, is the one Pare uses for all of them. In this notebook you implement an RTN linear layer, pack its codes two per byte, account for the memory, see where RTN fails, and check the implementation against Pare on a real model.

Prerequisite: Notebook 1.

In [ ]:
import sys

import matplotlib.pyplot as plt
import torch

sys.path.insert(0, "..")   # import Pare from this checkout; `pip install pare-quant` also works
torch.manual_seed(42)
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

## 1. The algorithm

For one linear layer with weight $W \in \mathbb{R}^{O \times I}$ and group size $g$, RTN does the following for every row $i$ and every group of $g$ consecutive columns.

1. Compute the asymmetric scale and zero-point of the group from its minimum and maximum (Notebook 1, Section 3).
2. Round each weight to its nearest code, $q = \mathrm{clamp}(\lfloor w/s \rceil + z,\, 0,\, 2^b - 1)$.
3. Store the codes, the scale and the zero-point.

At inference the layer dequantizes $\hat{W} = s\,(q - z)$ and computes $x \hat{W}^T$ in FP16. Nothing in the procedure looks at activations, so the error in each weight is independent of how much that weight matters to the output.

In [ ]:
def grid(bits):
    return (-128, 127) if bits == 8 else (0, 2**bits - 1)


def group_scales(W, bits, group_size):
    # Asymmetric per-group scale and zero-point, as pare.core.scale.compute_scale computes them.
    q_min, q_max = grid(bits)
    n_out, n_in = W.shape
    Wg = W.reshape(n_out, n_in // group_size, group_size)
    w_min = Wg.amin(dim=2, keepdim=True).clamp(max=0.0)
    w_max = Wg.amax(dim=2, keepdim=True).clamp(min=0.0)
    s = ((w_max - w_min) / (q_max - q_min)).clamp(min=1e-8)
    z = torch.round(q_min - w_min / s).clamp(q_min, q_max)
    return s, z                                        # both [n_out, n_groups, 1]


class RTNLinear(torch.nn.Module):
    def __init__(self, linear: torch.nn.Linear, bits: int = 4, group_size: int = 128):
        super().__init__()
        W = linear.weight.detach().float()
        self.bits, self.group_size = bits, group_size
        self.n_out, self.n_in = W.shape
        q_min, q_max = grid(bits)
        s, z = group_scales(W, bits, group_size)
        Wg = W.reshape(self.n_out, -1, group_size)
        self.q = torch.clamp(torch.round(Wg / s + z), q_min, q_max).to(torch.int32)
        self.s, self.z = s, z
        self.bias = None if linear.bias is None else linear.bias.detach()

    def dequantize(self):
        return (self.s * (self.q.float() - self.z)).reshape(self.n_out, self.n_in)

    def forward(self, x):
        return torch.nn.functional.linear(x, self.dequantize().to(x.dtype), self.bias)


linear = torch.nn.Linear(512, 256)
rtn4 = RTNLinear(linear, bits=4, group_size=128)
rtn8 = RTNLinear(linear, bits=8, group_size=512)     # one group per row: per-channel INT8
W = linear.weight.detach()
print(f"INT8 weight MAE {(rtn8.dequantize() - W).abs().mean():.5f}")
print(f"INT4 weight MAE {(rtn4.dequantize() - W).abs().mean():.5f}")
print(f"INT4 scales: {tuple(rtn4.s.shape)}, one per group of {rtn4.group_size}")

## 2. Weight error is not output error

The weight MAE says how far the stored values sit from the originals. What the model sees is the output $x W^T$, and the two errors differ whenever the input channels differ in magnitude: an error on a weight that multiplies a large input moves the output more than the same error on a weight that multiplies a small one. GPTQ and AWQ both start from this observation.

In [ ]:
x = torch.randn(32, 512)
with torch.no_grad():
    y = linear(x)
    rel = {name: ((y - layer(x)).norm() / y.norm()).item()
           for name, layer in [("INT8", rtn8), ("INT4", rtn4)]}
for name, r in rel.items():
    print(f"{name} relative output error {r:.4f}")

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for ax, (label, yy) in zip(axes, [("FP32", y), ("RTN INT8", rtn8(x)), ("RTN INT4", rtn4(x))]):
    ax.hist(yy.detach().flatten().numpy(), bins=60, alpha=0.8)
    ax.set_title(label)
    ax.set_xlabel("output value")
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

With Gaussian inputs of equal scale on every channel, the relative output error is about the relative weight error. Section 5 constructs an input where they differ by a large factor.

## 3. Packing INT4 codes

An INT4 code stored in an `int8` wastes half the byte. Pare packs two codes per byte with the code at an even column in the low nibble and the code at the next odd column in the high nibble. Because the codes are unsigned, packing and unpacking are plain masks and shifts with no sign extension.

In [ ]:
from pare.core.pack import pack_int4 as pare_pack
from pare.core.pack import unpack_int4 as pare_unpack


def pack_int4(q):
    q = q.to(torch.int32)
    low = q[..., 0::2] & 0xF
    high = (q[..., 1::2] & 0xF) << 4
    return (low | high).to(torch.uint8)


def unpack_int4(packed):
    p = packed.to(torch.int32)
    low, high = p & 0xF, (p >> 4) & 0xF
    return torch.stack([low, high], dim=-1).reshape(*packed.shape[:-1], -1)


q_codes = rtn4.q.reshape(rtn4.n_out, rtn4.n_in)      # [256, 512] codes in [0, 15]
packed = pack_int4(q_codes)
assert torch.equal(unpack_int4(packed), q_codes)
print(f"codes {tuple(q_codes.shape)} int32 -> packed {tuple(packed.shape)} uint8")
print(f"first row, first 8 codes: {q_codes[0, :8].tolist()}")
print(f"first row, first 4 bytes: {[f'0x{b:02x}' for b in packed[0, :4].tolist()]}")

# Pare's pack_int4 and unpack_int4 use the same layout.
assert torch.equal(pare_pack(q_codes), packed)
assert torch.equal(pare_unpack(packed), q_codes)
print("packing matches pare.core.pack")

## 4. Memory accounting

Per weight, INT4 storage costs 4 bits for the code plus the scale and zero-point shared by its group. Pare keeps the scale and the zero-point in FP32, so a group of 128 weights carries 8 bytes of metadata, or 0.0625 bytes per weight. For a $4096 \times 4096$ layer that gives the following.

In [ ]:
def memory_mb(n_out, n_in, group_size=128, meta_bytes=8):
    fp16 = n_out * n_in * 2
    codes = n_out * n_in // 2
    meta = n_out * (n_in // group_size) * meta_bytes
    return fp16 / 1e6, codes / 1e6, meta / 1e6


for shape in [(4096, 4096), (4096, 11008)]:
    fp16, codes, meta = memory_mb(*shape)
    print(f"{shape[0]} x {shape[1]}: FP16 {fp16:.1f} MB, INT4 codes {codes:.1f} MB, "
          f"FP32 scale and zero {meta:.2f} MB, ratio {fp16 / (codes + meta):.2f}x")

The ratio is 3.6 rather than 4 because of the metadata. Storing the scale and zero-point in FP16 would raise it to 3.8, and a group size of 32 would lower it to 3.0; the KV cache in Pare, which uses groups of 32, pays that price for accuracy.

## 5. Where RTN fails

A few weights in an LLM carry large magnitudes and multiply input channels with large activations. In a group that contains one such weight, the maximum sets the step, every other weight in the group is rounded on a coarse grid, and the group's whole contribution to the output degrades. The cell below shows the effect with one outlier weight on a channel with a large input.

In [ ]:
torch.manual_seed(1)
w_plain = torch.randn(1, 128) * 0.05
w_outlier = w_plain.clone()
w_outlier[0, 64] = 0.8                      # one large weight

x = torch.ones(128)
x[64] = 10.0                                # and a large input on the same channel

for W_row, label in [(w_plain, "no outlier"), (w_outlier, "one outlier")]:
    s, z = group_scales(W_row, 4, 128)
    q = torch.clamp(torch.round(W_row.reshape(1, 1, 128) / s + z), 0, 15)
    W_hat = (s * (q - z)).reshape(1, 128)
    out, out_hat = (W_row * x).sum(), (W_hat * x).sum()
    others = torch.arange(128) != 64
    print(f"{label:>12}: step {s.item():.4f}, MAE of the other 127 weights "
          f"{(W_hat - W_row)[0, others].abs().mean():.5f}, output {out:.3f} -> {out_hat:.3f}")

The outlier makes the step about three times larger, and the rounding error of the other 127 weights grows by the same factor, although none of them changed. The three calibrated schemes respond differently. GPTQ compensates each rounding error by adjusting the weights not yet quantized (Notebook 3). AWQ scales salient channels up before rounding and folds the inverse into the previous layer (Notebook 4). SmoothQuant moves the dynamic range of the activations into the weights, where it is easier to quantize (Notebook 5).

## 6. RTN in Pare on GPT-2

`QuantConfig(scheme="rtn")` applies the procedure above to every `nn.Linear` in a model (GPT-2's `Conv1D` layers are converted first). The cell checks that Pare's layer and `RTNLinear` produce the same dequantized weights, then measures WikiText-2 perplexity for FP32, RTN INT8 per channel and RTN INT4 with groups of 128. The evaluation uses 20 windows of 512 tokens so that it runs on a CPU in about a minute; the README reports the full test set for 7B models.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

from pare import QuantConfig, quantize
from pare.eval.perplexity import evaluate_perplexity
from pare.layers.linear import QuantizedLinear

# 1. Pare's RTN layer equals RTNLinear.
linear = torch.nn.Linear(512, 256)
cfg = QuantConfig(bits=4, scheme="rtn", granularity="per_group", group_size=128)
assert torch.allclose(QuantizedLinear.from_linear(linear, cfg).dequantize(),
                      RTNLinear(linear, 4, 128).dequantize())
print("QuantizedLinear.from_linear matches RTNLinear")

# 2. Perplexity on GPT-2.
model_id = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_id)


def fresh_model():
    return AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float32)


def ppl(model):
    return evaluate_perplexity(
        model, tokenizer, dataset="wikitext2", seq_len=512, n_samples=20, device="cpu"
    )


results = {"FP32": ppl(fresh_model())}
configs = {
    "RTN INT8 per channel": QuantConfig(bits=8, scheme="rtn", granularity="per_channel"),
    "RTN INT4 g=128": QuantConfig(bits=4, scheme="rtn", granularity="per_group", group_size=128),
}
for label, config in configs.items():
    model = fresh_model()
    quantize(model, config)
    results[label] = ppl(model)

for label, value in results.items():
    print(f"{label:>22}: PPL {value:6.2f}  ({value - results['FP32']:+.2f})")

The printed deltas are the whole story of RTN. INT8 moves perplexity by a fraction of a point, while INT4 adds several points on this 124M-parameter model. On the 7B models in the README, RTN INT8 stays within 0.01 perplexity of FP16; RTN INT4 is not in those tables, and the INT4 rows there come from GPTQ and AWQ.

## Exercises

1. In Section 6, add `QuantConfig(bits=4, scheme="rtn", granularity="per_channel")` and `group_size=32` to the comparison. How does perplexity move with the group size, and what does each setting cost in metadata (Section 4)?
2. Change `RTNLinear` to use the symmetric scale from Notebook 1 and repeat the Section 6 equivalence check with `sym=True` in the config.
3. In Section 5, move the large input to a channel without the outlier weight. Why does the output error fall even though the weight MAE does not change?

## Summary

| | RTN INT8 | RTN INT4 |
|---|---|---|
| Calibration data | none | none |
| Storage per weight | 1 byte plus per-row scale | 0.5 byte plus per-group scale and zero-point |
| 7B perplexity change (README) | at most 0.01 | not benchmarked; GPTQ and AWQ are used at INT4 |
| Failure mode | | one large weight sets the step for its whole group |

Notebook 3 introduces GPTQ, which keeps RTN's storage format and reduces the output error by adjusting the weights that have not yet been rounded.